In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/1h-dataset-and-process/0.2c Matched with scimago.xlsx
/kaggle/input/1h-dataset-and-process/0.3a 1H records selection-2.xlsx
/kaggle/input/1h-dataset-and-process/2.3 Semantic trends.xlsx
/kaggle/input/1h-dataset-and-process/2.1a Keywords BERTopic_Results.xlsx
/kaggle/input/1h-dataset-and-process/2.1 Keywords_BioBERT.xlsx
/kaggle/input/1h-dataset-and-process/3.2 Citation Analysis_remove later.xlsx
/kaggle/input/1h-dataset-and-process/2.2a Clusters_Weighted.xlsx
/kaggle/input/1h-dataset-and-process/2.1 Keywords_TFIDF (low coherence_for comparison).xlsx
/kaggle/input/1h-dataset-and-process/1.2 Extracted Country.xlsx
/kaggle/input/1h-dataset-and-process/0.3a 1H records selection-1.xlsx
/kaggle/input/1h-dataset-and-process/2.3 Semantic trends_weighted.xlsx
/kaggle/input/1h-dataset-and-process/2.1a Keywords BERTopic_Results_Weighted_14Topics.xlsx
/kaggle/input/1h-dataset-and-process/0.1d Duplicate Removal.xlsx
/kaggle/input/1h-dataset-and-process/0.2b Extracted ISSN.xlsx
/kaggle

In [2]:
# 0.1 Data Preprocessing, Merging datasets

# Convert to 0-based index
def column_to_index(column_letter):
    index = 0
    for char in column_letter:
        index = index * 26 + (ord(char.upper()) - ord('A') + 1)
    return index - 1

# Define WoS and PubMed column names
wos_columns = [
    'Publication Type', 'Authors', 'Book Authors', 'Book Editors', 'Book Group Authors',
    'Author Full Names', 'Book Author Full Names', 'Group Authors', 'Article Title',
    'Source Title', 'Book Series Title', 'Book Series Subtitle', 'Language',
    'Document Type', 'Conference Title', 'Conference Date', 'Conference Location',
    'Conference Sponsor', 'Conference Host', 'Author Keywords', 'Keywords Plus',
    'Abstract', 'Addresses', 'Affiliations', 'Reprint Addresses', 'Email Addresses',
    'Researcher Ids', 'ORCIDs', 'Funding Orgs', 'Funding Name Preferred',
    'Funding Text', 'Cited References', 'Cited Reference Count',
    'Times Cited, WoS Core', 'Times Cited, All Databases', '180 Day Usage Count',
    'Since 2013 Usage Count', 'Publisher', 'Publisher City', 'Publisher Address',
    'ISSN', 'eISSN', 'ISBN', 'Journal Abbreviation', 'Journal ISO Abbreviation',
    'Publication Date', 'Publication Year', 'Volume', 'Issue', 'Part Number',
    'Supplement', 'Special Issue', 'Meeting Abstract', 'Start Page', 'End Page',
    'Article Number', 'DOI', 'DOI Link', 'Book DOI', 'Early Access Date',
    'Number of Pages', 'WoS Categories', 'Web of Science Index', 'Research Areas',
    'IDS Number', 'Pubmed Id', 'Open Access Designations', 'Highly Cited Status',
    'Hot Paper Status', 'Date of Export', 'UT (Unique WOS ID)', 'Web of Science Record'
]

pubmed_columns = [
    'PMID (PubMed ID)', 'OWN (Owner)', 'STAT (Status)', 'DCOM (Date Completed)',
    'LR (Last Revised)', 'IS (ISSN)', 'VI (Volume)', 'IP (Issue)',
    'DP (Publication Date)', 'TI (Title)', 'PG (Pages)', 'LID (Location ID)',
    'AB (Abstract)', 'CN (Corporate Author)', 'FAU (Full Author)', 'AU (Author)',
    'AD (Affiliation)', 'LA (Language)', 'GR (Grant Number)', 'PT (Publication Type)',
    'PL (Place of Publication)', 'TA (Journal Abbreviation)', 'JT (Journal Title)',
    'JID (Journal ID)', 'SB (Subset)', 'MH (MeSH Terms)', 'PMC (PMC ID)',
    'EDAT (Entrez Date)', 'MHDA (MeSH Date)', 'PMCR (PMC Release)',
    'CRDT (Create Date)', 'PHST (Publication History)', 'AID (Article ID)',
    'PST (Publication Status)', 'SO (Source)', 'AUID (Author ID)',
    'DEP (Deposit Date)', 'OT (Other Terms)', 'RN (Registry Number)',
    'PMCID (PMC Crossref ID)', 'PUBMED (PubMed Secondary ID)', 'DA (Date Created)',
    'NM (Substance Name)', 'PS (Personal Subject)', 'SFM (Spaceflight Mission)'
]

# Define WoS-PM column letter mappings
wos_pm_pairs = [
    ('A', 'T'), ('B', 'P'), ('C', ''), ('D', ''), ('E', ''), ('F', 'O'),
    ('G', ''), ('H', ''), ('I', 'J'), ('J', 'W'), ('K', ''), ('L', ''),
    ('M', 'R'), ('N', 'T'), ('O', ''), ('P', ''), ('Q', ''), ('R', ''),
    ('S', ''), ('T', 'Z'), ('U', 'Z'), ('V', 'M'), ('W', 'Q'), ('X', 'Q'),
    ('Y', ''), ('Z', ''), ('AA', ''), ('AB', 'AJ'), ('AC', 'S'), ('AD', 'S'),
    ('AE', ''), ('AF', ''), ('AG', ''), ('AH', ''), ('AI', ''), ('AJ', ''),
    ('AK', ''), ('AL', ''), ('AM', 'U'), ('AN', ''), ('AO', 'F'), ('AP', ''),
    ('AQ', ''), ('AR', ''), ('AS', ''), ('AT', 'I'), ('AU', 'I'), ('AV', 'G'),
    ('AW', 'H'), ('AX', ''), ('AY', ''), ('AZ', ''), ('BA', ''), ('BB', ''),
    ('BC', ''), ('BD', 'AG'), ('BE', 'L'), ('BF', ''), ('BG', ''), ('BH', ''),
    ('BI', 'K'), ('BJ', ''), ('BK', ''), ('BL', ''), ('BM', ''), ('BN', 'A'),
    ('BO', ''), ('BP', ''), ('BQ', ''), ('BR', ''), ('BS', ''), ('BT', '')
]

# Create mapping dictionary with multiple targets
mapping_dict = {}
for wos_letter, pm_letter in wos_pm_pairs:
    if not pm_letter:
        continue
    
    wos_col = wos_columns[column_to_index(wos_letter)]
    pm_col = pubmed_columns[column_to_index(pm_letter)]
    
    if pm_col not in mapping_dict:
        mapping_dict[pm_col] = []
    mapping_dict[pm_col].append(wos_col)

# Load datasets
wos_df = pd.read_excel('/kaggle/input/1h-dataset-and-process/0.1b WoS_790 31Dec2024 savedrecs.xlsx')
pubmed_df = pd.read_excel('/kaggle/input/1h-dataset-and-process/0.1a Pubmed_998 31Dec2024.xlsx')

# Add source column
wos_df.insert(0, 'Source', 'WoS')
pubmed_df.insert(0, 'Source', 'PubMed')

# Process PubMed columns - copy to all mapped WoS columns
for pm_col, wos_cols in mapping_dict.items():
    for wos_col in wos_cols:
        pubmed_df[wos_col] = pubmed_df[pm_col]

# Get all WoS columns (excluding Source)
wos_column_order = wos_df.columns.tolist()[1:]

# Get PubMed-specific columns (those not in WoS columns)
pubmed_unmatched = [col for col in pubmed_df.columns 
                    if col not in wos_column_order and col != 'Source']

# Create final column order
desired_columns = ['Source'] + wos_column_order + pubmed_unmatched

# Merge datasets
merged_df = pd.concat([wos_df, pubmed_df], axis=0, ignore_index=True)[desired_columns]

# Save to Excel
output_path = '0.1c Merged.xlsx'
merged_df.to_excel(output_path, index=False)
print(f'Processed file saved to: {output_path}')

Processed file saved to: 0.1c Merged.xlsx


In [3]:
# 0.1 Data Preprocessing, Duplicate Removal
from tqdm import tqdm # to show progress in large datasets
from collections import defaultdict, Counter

# Load dataset
input_path = '/kaggle/input/1h-dataset-and-process/0.1c Merged 1788.xlsx'
output_path = '/kaggle/working/0.1d Duplicate Removal.xlsx'

print("Loading dataset...")
df = pd.read_excel(input_path)

# Store original row number (start from 2), only for duplicates sheet
df['Original Row Number'] = df.index + 2

# Columns to check for duplication
check_cols = ['Pubmed Id', 'Article Title', 'Abstract']

# Create temp columns to handle NaN and detect duplicates
for col in check_cols:
    df[f'{col}_temp'] = df[col].fillna(df.index.to_series().apply(lambda x: f'NaN_{x}'))

# Track first occurrence of each value
first_occurrence_map = {col: {} for col in check_cols}
remove_indices = set()
removal_reasons = defaultdict(list)
removal_reason_counter = Counter()

print("Checking for duplicates (removing Preprints)...")
for col in tqdm(check_cols):
    temp_col = f'{col}_temp'
    for idx, val in df[temp_col].items():
        if val in first_occurrence_map[col]:
            first_idx = first_occurrence_map[col][val]
            first_row_number = df.at[first_idx, 'Original Row Number']
            current_row_number = df.at[idx, 'Original Row Number']

            pub_type_first = str(df.at[first_idx, 'Publication Type']).strip().lower()
            pub_type_current = str(df.at[idx, 'Publication Type']).strip().lower()

            if pub_type_current == 'preprint':
                remove_indices.add(idx)
                removal_reasons[idx].append(f'Duplicated {col} with Row {first_row_number}')
                removal_reason_counter[col] += 1
            elif pub_type_first == 'preprint':
                remove_indices.add(first_idx)
                removal_reasons[first_idx].append(f'Duplicated {col} with Row {current_row_number}')
                removal_reason_counter[col] += 1
                first_occurrence_map[col][val] = idx
            else:
                remove_indices.add(idx)
                removal_reasons[idx].append(f'Duplicated {col} with Row {first_row_number}')
                removal_reason_counter[col] += 1
        else:
            first_occurrence_map[col][val] = idx

# Build duplicates DataFrame
duplicates_df = df.loc[list(remove_indices)].copy()
duplicates_df.insert(1, 'Reason for Removal', duplicates_df.index.map(lambda x: '; '.join(removal_reasons[x])))

# Move 'Original Row Number' and 'Reason for Removal' to the start
start_cols = ['Original Row Number', 'Reason for Removal']
other_cols = [col for col in duplicates_df.columns if col not in start_cols]
duplicates_df = duplicates_df[start_cols + other_cols]

# Build cleaned dataset (remove 'Original Row Number' column before export)
cleaned_df = df.drop(remove_indices).drop(columns=['Original Row Number'], errors='ignore').copy()

# Drop temp columns from all dataframes
for col in check_cols:
    temp_col = f'{col}_temp'
    for d in [df, cleaned_df, duplicates_df]:
        d.drop(columns=temp_col, inplace=True, errors='ignore')

# Save to a single Excel file
print("Saving results...")
with pd.ExcelWriter(output_path) as writer:
    cleaned_df.to_excel(writer, sheet_name='Cleaned Records', index=False)
    duplicates_df.to_excel(writer, sheet_name='Removed Duplicates', index=False)

# Summary
print(f"\nSummary:")
print(f"Original records: {len(df)}")
print(f"Cleaned records: {len(cleaned_df)}")
print(f"Duplicates removed: {len(duplicates_df)}")
print("\nBreakdown of duplicates removed by column:")
for col in check_cols:
    print(f"- {col}: {removal_reason_counter[col]} duplicates removed")

# Breakdown of removed duplicates by 'Source'
if 'Source' in duplicates_df.columns:
    print("\nBreakdown of removed duplicates by Source:")
    source_counts = duplicates_df['Source'].value_counts(dropna=False)
    for source, count in source_counts.items():
        print(f"- {source}: {count}")


print(f'Processed file saved to: {output_path}')

Loading dataset...
Checking for duplicates (removing Preprints)...


100%|██████████| 3/3 [00:00<00:00, 170.80it/s]

Saving results...



Summary:
Original records: 1788
Cleaned records: 1312
Duplicates removed: 476

Breakdown of duplicates removed by column:
- Pubmed Id: 472 duplicates removed
- Article Title: 13 duplicates removed
- Abstract: 125 duplicates removed

Breakdown of removed duplicates by Source:
- PubMed: 476
Processed file saved to: /kaggle/working/0.1d Duplicate Removal.xlsx


In [4]:
# 0.2 Matching and Weighing, extracting ISSN
input_path = '/kaggle/input/1h-dataset-and-process/0.1d Duplicate Removal.xlsx'
output_path = '/kaggle/working/0.2b Extracted ISSN.xlsx'

# Load only 'Cleaned Records' sheet
df = pd.read_excel(input_path, sheet_name='Cleaned Records')

# Regex pattern to match ISSN-like strings (e.g., 1234-5678)
issn_pattern = r'(\b[A-Za-z0-9]{4}-[A-Za-z0-9]{4}\b)'

# Find exact column names for 'ISSN' and 'eISSN' (case-insensitive match, keep original name)
issn_col = next((col for col in df.columns if col.strip().lower() == 'issn'), None)
eissn_col = next((col for col in df.columns if col.strip().lower() == 'eissn'), None)

# Check if both columns are found
if issn_col is None or eissn_col is None:
    raise ValueError("Either 'ISSN' or 'eISSN' column is missing in 'Cleaned Records'.")

# Extract ISSN patterns from both columns
df['issn_matches'] = df[issn_col].astype(str).str.extractall(issn_pattern).groupby(level=0).agg(', '.join)
df['eissn_matches'] = df[eissn_col].astype(str).str.extractall(issn_pattern).groupby(level=0).agg(', '.join)

# Combine into a single column
df['Extracted ISSN'] = df['issn_matches'].fillna('') + ', ' + df['eissn_matches'].fillna('')
df['Extracted ISSN'] = df['Extracted ISSN'].str.strip(', ')

# Drop temporary match columns
df.drop(columns=['issn_matches', 'eissn_matches'], inplace=True)

# Save the result
df.to_excel(output_path, sheet_name='Cleaned Records', index=False)

print(f'Processed file saved to: {output_path}')

Processed file saved to: /kaggle/working/0.2b Extracted ISSN.xlsx


In [5]:
# 0.2 Matching and Weighing, matching ISSN with scimago data
import re # regular expressions

# Load datasets
dataset1 = pd.read_excel('/kaggle/input/1h-dataset-and-process/0.2b Extracted ISSN.xlsx')
dataset2 = pd.read_excel('/kaggle/input/1h-dataset-and-process/0.2a scimago_2023.xlsx')

# Normalize ISSNs: remove non-alphanum, standardize to XXXX-XXXX
def normalize_issn(issn):
    issn = re.sub(r'[^0-9Xx]', '', str(issn))
    if len(issn) == 8:
        return issn[:4] + '-' + issn[4:].upper()
    return None

def preprocess_issn_column(issn_column):
    return issn_column.fillna('').astype(str).str.split(',').apply(
        lambda lst: set(filter(None, [normalize_issn(i.strip()) for i in lst]))
    )

# Apply normalization
dataset1['ISSN_set'] = preprocess_issn_column(dataset1['Extracted ISSN'])
dataset2['ISSN_set'] = preprocess_issn_column(dataset2['Issn'])

# Match ISSNs
matches = []
for idx1, row1 in dataset1.iterrows():
    issn_set1 = row1['ISSN_set']
    match = dataset2[dataset2['ISSN_set'].apply(lambda x: not issn_set1.isdisjoint(x))]
    
    if not match.empty:
        matched_issns = ', '.join(sorted(issn_set1.intersection(match.iloc[0]['ISSN_set'])))
        combined_row = row1.tolist() + [matched_issns] + match.iloc[0].tolist()
    else:
        combined_row = row1.tolist() + [None] + [''] * len(dataset2.columns)
    
    matches.append(combined_row)

# Create result DataFrame
results_columns = dataset1.columns.tolist() + ['Matched ISSNs'] + dataset2.columns.tolist()
results_df = pd.DataFrame(matches, columns=results_columns)

# Save to Excel
output_path = '0.2c Matched with scimago.xlsx'
results_df.to_excel(output_path, index=False)
print(f'Processed file saved to: {output_path}')

Processed file saved to: 0.2c Matched with scimago.xlsx


In [6]:
# 0.2 Preview Matched data (no 1H selection yet)
dataset_path = '/kaggle/input/1h-dataset-and-process/0.2c Matched with scimago.xlsx'
df = pd.read_excel(dataset_path)

# Use 'Title' instead of 'Source Title'
grouped_counts = df['Title'].value_counts().reset_index()
grouped_counts.columns = ['Source Title', 'Frequency']
grouped_counts['Percentage'] = (grouped_counts['Frequency'] / grouped_counts['Frequency'].sum()) * 100

# Extract relevant columns — ensure no duplicates, group by Title to ensure consistent SJR per journal
sjr_cols = df.groupby('Title', as_index=False).agg({
    'SJR': 'first',  # Take the first SJR value per journal
    'SJR Best Quartile': 'first',
    'H index': 'first'
})

# Merge counts with SJR data
merged_counts = pd.merge(
    grouped_counts,
    sjr_cols,
    left_on='Source Title',
    right_on='Title',
    how='left'
).drop('Title', axis=1)

# Format Percentage as ##.##%
merged_counts['Percentage'] = merged_counts['Percentage'].apply(lambda x: f'{x:.2f}%')

# Convert SJR to numeric, replace ',' with '.', handle NaNs, and apply the correct format
merged_counts['SJR'] = merged_counts['SJR'].apply(lambda x: str(x).replace(',', '.') if pd.notna(x) else x)
merged_counts['SJR'] = pd.to_numeric(merged_counts['SJR'], errors='coerce')
merged_counts['SJR'] = merged_counts['SJR'].apply(lambda x: f'{x:.3f}' if pd.notna(x) else '')

# Add total row — exclude empty entries before concat
total_row = pd.DataFrame({
    'Source Title': ['Total'],
    'Frequency': [merged_counts['Frequency'].sum()],
    'Percentage': ['100.00%'],
    'SJR': [''],  # Avoid pd.NA to prevent FutureWarning
    'SJR Best Quartile': [''],
    'H index': ['']
})

# Concatenate all records and total row
final_df = pd.concat([merged_counts, total_row], ignore_index=True)

# Fill NaNs selectively
final_df['SJR Best Quartile'] = final_df['SJR Best Quartile'].fillna('')
final_df['H index'] = final_df['H index'].fillna('')

# Adjust index to start from 1
final_df.index = final_df.index + 1

# Display results
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

print("Source Titles with Frequency, Percentage, SJR, SJR Best Quartile, and H index:")
display(pd.concat([final_df.head(20), final_df.tail(5)]))
print(f"Total number of unique source titles: {len(merged_counts)}")

Source Titles with Frequency, Percentage, SJR, SJR Best Quartile, and H index:


,Source Title,Frequency,Percentage,SJR,SJR Best Quartile,H index
1,One Health,101,8.02%,0.971,Q1,39.0
2,Veterinary World,57,4.52%,0.483,Q2,48.0
3,PLoS ONE,37,2.94%,0.839,Q1,435.0
4,Frontiers in Veterinary Science,34,2.70%,0.734,Q1,70.0
5,Scientific Reports,22,1.75%,0.900,Q1,315.0
6,Acta Tropica,21,1.67%,0.708,Q1,117.0
7,Frontiers in Public Health,21,1.67%,0.895,Q1,101.0
8,Frontiers in Microbiology,21,1.67%,1.065,Q1,233.0
9,Antibiotics,19,1.51%,0.920,Q1,77.0
10,PLoS Neglected Tropical Diseases,19,1.51%,1.258,Q1,163.0


Total number of unique source titles: 408


In [7]:
# 0.2 Preview Matched data (plotly version, no 1H selection yet)
import plotly.graph_objects as go

# Prepare the combined DataFrame for display (top 20 + last 5 rows)
display_df = pd.concat([final_df.head(20), final_df.tail(5)])

# Build Plotly Table
fig = go.Figure(data=[go.Table(
    columnwidth=[200] + [80]*(len(display_df.columns)-1),
    header=dict(
        values=list(display_df.columns),
        fill_color='lightblue',
        align='left',
        font=dict(color='black', size=12)
    ),
    cells=dict(
        values=[display_df[col].tolist() for col in display_df.columns],
        fill_color='white',
        align='left',
        font=dict(color='black', size=11)
    )
)])

# Adjust layout size
fig.update_layout(
    width=1000,
    height=700,
    margin=dict(l=0, r=0, t=30, b=0)
)

# Show table
fig.show()

In [8]:
# 0.3 1H records selection, install scikit
#!pip install -U scikit-learn imbalanced-learn # install
!pip install scikit-learn imbalanced-learn > /dev/null 2>&1 #suppress long output

In [9]:
# 0.3 1H records selection-1, process and metrics
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

# Load dataset
df = pd.read_excel("/kaggle/input/1h-dataset-and-process/0.2c Matched with scimago.xlsx")

# Corrected fallback columns
fallback_columns = ['Article Title', 'Author Keywords', 'Keywords Plus', 'Abstract', 'Title']

# Define GSEA and excluded countries
gsea_countries = {
    "Brunei", "Cambodia", "Indonesia", "Laos", "Malaysia", "Myanmar", 
    "Philippines", "Singapore", "Thailand", "Vietnam", "Taiwan", "Timor-Leste", 
    "Southeast Asia", "Western Pacific"
}

excluded_countries = {
    "India", "Uganda", "Africa", "Suriname", "Hong Kong", "Tuvalu", "China", "Japan", "Germany",
    "Pakistan", "Southern Africa", "UK", "United States", "Australia", "France", "Mongolia",
    "St. Kitts", "Tanzania", "Brazil", "Russia", "South Korea", "Mexico", "Egypt", "Italy",
    "Argentina", "Canada", "New Zealand", "Saudi Arabia", "Turkey", "South Africa"
}

# Functions
def is_one_health_related(text):
    keywords = [
        "zoonotic", "AMR", "vector-borne", "climate", "environmental", "biodiversity", "urbanization",
        "non-communicable diseases", "food safety", "food security", "antimicrobial resistance",
        "infectious diseases", "one health", "policy", "bacteria", "virus", "parasite", "animal study",
        "public health", "epidemiology"
    ]
    return any(re.search(rf"\b{kw}\b", str(text), re.IGNORECASE) for kw in keywords)

def find_gsea_country(text):
    return next((country for country in gsea_countries if re.search(rf"\b{country}\b", str(text), re.IGNORECASE)), None)

def find_excluded_country(text):
    return next((country for country in excluded_countries if re.search(rf"\b{country}\b", str(text), re.IGNORECASE)), None)

def is_bibliometric_study(text):
    keywords = ["bibliometric", "scientometric", "citation analysis", "co-authorship", "co-citation"]
    return any(re.search(rf"\b{kw}\b", str(text), re.IGNORECASE) for kw in keywords)

# Combine all fallback columns into one text field
df["combined_text"] = df[fallback_columns].apply(lambda x: ' '.join(str(val) for val in x if pd.notnull(val)), axis=1)

# Create flags
df["one_health_related"] = df[fallback_columns].apply(lambda x: any(is_one_health_related(val) for val in x if pd.notnull(val)), axis=1)
df["region"] = df[fallback_columns].apply(lambda x: next((find_gsea_country(val) for val in x if find_gsea_country(val)), None), axis=1)
df["excluded_country"] = df[fallback_columns].apply(lambda x: next((find_excluded_country(val) for val in x if find_excluded_country(val)), None), axis=1)
df["bibliometric_study"] = df[fallback_columns].apply(lambda x: any(is_bibliometric_study(val) for val in x if pd.notnull(val)), axis=1)

# Determine initial inclusion
df["included"] = (df["one_health_related"] | df["region"].notnull()) & ~df["bibliometric_study"] & df["excluded_country"].isnull()

# Reason for inclusion/exclusion
def get_inclusion_reason(row):
    if row["included"]:
        return f"Meets One Health criteria - {row['region']}" if row["region"] else "ML Model Inclusion with Unclear Region"
    if row["bibliometric_study"]:
        return "Bibliometric Study"
    if not row["one_health_related"]:
        return "Not One Health Related"
    if row["excluded_country"]:
        return f"Excluded Country: {row['excluded_country']}"
    return "Not in Southeast Asia or Taiwan"

df["reason"] = df.apply(get_inclusion_reason, axis=1)

# Track original row numbers, starting from 2
df.reset_index(inplace=True)
df.rename(columns={"index": "original_row"}, inplace=True)
df["original_row"] += 2

# Machine Learning Section
X = df["combined_text"]
y = df["included"]

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
X_tfidf = vectorizer.fit_transform(X)

# Handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_tfidf, y)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled
)

# Build a pipeline
pipeline = Pipeline([
    ("classifier", RandomForestClassifier(random_state=42))
])

# Hyperparameter tuning
param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10, 20],
    "classifier__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring='accuracy', verbose=1, n_jobs=-1
)
grid_search.fit(X_train, y_train)

# Model evaluation
best_pipeline = grid_search.best_estimator_
y_pred = best_pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

# Predict entire dataset
df["ml_predicted_included"] = best_pipeline.predict(X_tfidf)

# Final inclusion decision
df["final_included"] = df["included"] | df["ml_predicted_included"]
df["final_reason"] = df.apply(
    lambda row: "ML Model Inclusion" if row["ml_predicted_included"] and not row["included"] else row["reason"],
    axis=1
)

# Additional Inclusion: Include all records from GSEA countries (non-null region) unless explicitly marked "Not One Health Related"
df.loc[
    (df["region"].notnull()) & (df["final_reason"] != "Not One Health Related"),
    ["final_included", "final_reason"]
] = [True, "GSEA Region Inclusion"]

# Output
ordered_cols = ["original_row", "final_reason", "region"] + fallback_columns
additional_cols = [col for col in df.columns if col not in ordered_cols + [
    "included", "one_health_related", "bibliometric_study",
    "ml_predicted_included", "excluded_country", "final_included"
]]
output_columns = ordered_cols + additional_cols

# Move combined_text and reason after first 8 columns
first_8 = output_columns[:8]
reordered = first_8 + ["combined_text", "reason"] + [col for col in output_columns if col not in first_8 + ["combined_text", "reason"]]

included_df = df[df["final_included"]][reordered]
excluded_df = df[~df["final_included"]][reordered]

# Save outputs
output_path = "0.3a 1H records selection-1.xlsx"
with pd.ExcelWriter(output_path) as writer:
    included_df.to_excel(writer, sheet_name="Included", index=False)
    excluded_df.to_excel(writer, sheet_name="Excluded", index=False)

# Print summary stats
print("\nIncluded Records by Final Reason:")
print(included_df["final_reason"].value_counts())
print("\nIncluded Records by Region:")
print(included_df["region"].value_counts(dropna=False))
print("\nExcluded Records by Final Reason:")
print(excluded_df["final_reason"].value_counts())
print("\nExcluded Records by Region:")
print(excluded_df["region"].value_counts(dropna=False))

# --- Summary 2: Source Counts ---
print("\n=== 'Source' Summary: Included ===")
included_source = included_df['Source'].value_counts(dropna=False).rename_axis('Source').reset_index(name='Count')
included_source.loc[len(included_source)] = ['Total', included_source['Count'].sum()]
print(included_source)

print("\n=== 'Source' Summary: Excluded ===")
excluded_source = excluded_df['Source'].value_counts(dropna=False).rename_axis('Source').reset_index(name='Count')
excluded_source.loc[len(excluded_source)] = ['Total', excluded_source['Count'].sum()]
print(excluded_source)

print(f"\n✅ Processed file saved to: {output_path}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
              precision    recall  f1-score   support

       False       1.00      0.80      0.89       192
        True       0.83      1.00      0.91       192

    accuracy                           0.90       384
   macro avg       0.92      0.90      0.90       384
weighted avg       0.92      0.90      0.90       384


Included Records by Final Reason:
final_reason
GSEA Region Inclusion                     694
ML Model Inclusion with Unclear Region    389
ML Model Inclusion                         24
Name: count, dtype: int64

Included Records by Region:
region
None               413
Thailand           227
Vietnam            101
Malaysia            80
Indonesia           65
Southeast Asia      49
Philippines         38
Taiwan              37
Cambodia            34
Singapore           25
Myanmar             16
Laos                14
Western Pacific      4
Brunei               2
Timor-Leste          2
Name: count, dtype:

In [10]:
# 0.3 1H records selection-2, process and metrics
file_path = "/kaggle/input/1h-dataset-and-process/0.3a 1H records selection-1.xlsx"
included_df = pd.read_excel(file_path, sheet_name="Included")
excluded_df = pd.read_excel(file_path, sheet_name="Excluded")

# Define exclusion criteria
exclude_final_reason = "ML Model Inclusion" # there are other ML Inclusion with Unclear region
exclude_title_keywords = ["Journal of Physics: Conference Series", 
                          "Adapted Physical Activity Quarterly", "Chemosphere", "Children",
                          "Clinical and Experimental Dental Research", "Ecography", "Insects",
                          "Indian Journal of Medical Research", "Journal of Nursing", 
                          "Science"] # current records here not 1H related
exclude_publication_types = [
    "Editorial", "Letter", "Study Guide, Book Chapter",
    "Clinical Trial, Journal Article, Multicenter Study",
    "Comparative Study, Journal Article",
    "Letter, Research Support, Non-U.S. Gov't",
    "Letter, Research Support, N.I.H., Extramural, Research Support, Non-U.S. Gov't, Research Support, U.S. Gov't, Non-P.H.S.",
    "Published Erratum", "Review, Book", "Study Guide, Book Chapter" # non-research records
]
exclude_reason = "Not One Health Related" #Obviously

# Create a mask for exclusions
mask_final_reason = included_df['final_reason'] == exclude_final_reason
mask_title_keywords = included_df['Title'].fillna('').str.contains('|'.join(exclude_title_keywords), case=False)
mask_publication_type = included_df['Publication Type'].isin(exclude_publication_types)
mask_reason = included_df['reason'] == exclude_reason

# Combine all exclusion criteria
exclusion_mask = mask_final_reason | mask_title_keywords | mask_publication_type | mask_reason

# Split included and excluded records
new_excluded = included_df[exclusion_mask]
new_included = included_df[~exclusion_mask]

# Append new exclusions to the existing 'Excluded' dataframe
updated_excluded = pd.concat([excluded_df, new_excluded], ignore_index=True)

# Save both sheets back into a new Excel file
output_path = "/kaggle/working/0.3a 1H records selection-2.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    new_included.to_excel(writer, sheet_name='Included', index=False)
    updated_excluded.to_excel(writer, sheet_name='Excluded', index=False)

# Set pandas display options
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

# --- Summary 1: Excluded Records Reasons ---
print("\n=== Summary of Excluded Records: Final Reason ===")
print(new_excluded['final_reason'].value_counts(dropna=False).rename_axis('Final Reason').reset_index(name='Count'))

print("\n=== Summary of Excluded Records: Title Match ===")
print(
    new_excluded['Title'].str.extract(f"({'|'.join(exclude_title_keywords)})", expand=False)
    .value_counts(dropna=False)
    .rename_axis('Title Match').reset_index(name='Count')
)

print("\n=== Summary of Excluded Records: Publication Type ===")
print(new_excluded['Publication Type'].value_counts(dropna=False).rename_axis('Publication Type').reset_index(name='Count'))

print("\n=== Summary of Excluded Records: Reason ===")
print(new_excluded['reason'].value_counts(dropna=False).rename_axis('Reason').reset_index(name='Count'))

# --- Summary 2: Source Counts ---
print("\n=== 'Source' Summary: Included ===")
included_source = new_included['Source'].value_counts(dropna=False).rename_axis('Source').reset_index(name='Count')
included_source.loc[len(included_source)] = ['Total', included_source['Count'].sum()]
print(included_source)

print("\n=== 'Source' Summary: Excluded ===")
excluded_source = updated_excluded['Source'].value_counts(dropna=False).rename_axis('Source').reset_index(name='Count')
excluded_source.loc[len(excluded_source)] = ['Total', excluded_source['Count'].sum()]
print(excluded_source)

print(f"\n✅ Processed file saved to: {output_path}")


=== Summary of Excluded Records: Final Reason ===
                             Final Reason  Count
0  ML Model Inclusion with Unclear Region     65
1                   GSEA Region Inclusion     62
2                      ML Model Inclusion     24

=== Summary of Excluded Records: Title Match ===
                                  Title Match  Count
0                                     Science     94
1                                         NaN     48
2          Indian Journal of Medical Research      1
3                                   Ecography      1
4                                     Insects      1
5       Journal of Physics: Conference Series      1
6                                    Children      1
7                                 Chemosphere      1
8                          Journal of Nursing      1
9   Clinical and Experimental Dental Research      1
10        Adapted Physical Activity Quarterly      1

=== Summary of Excluded Records: Publication Type ===
            